# Volt Notebook

Use Python to explore your simulations, analyses, and results on Volt.
This notebook is already connected to your Volt instance — just run the cells.

## Connect

In [ ]:
from voltsdk import VoltClient
import os

client = VoltClient.from_env()
print(f"Connected to team: {client.team.get('name')}")

## Load the trajectory

If you opened this notebook from a trajectory it is linked automatically;
otherwise pick one from the list.

In [ ]:
trajectory_id = os.environ.get('VOLT_TRAJECTORY_ID', '')

if trajectory_id:
    traj = client.trajectories.get(trajectory_id)
    print(f"Trajectory: {traj.raw.get('name')}")
    print(f"Status: {traj.raw.get('status')}")
    print(f"Frames: {len(traj.frames)}")
else:
    print('No trajectory linked. Available trajectories:')
    for t in client.trajectories.list():
        print(f"  {t.raw.get('name')} (id={t.id})")
    print("\nLoad one with: traj = client.trajectories.get('paste_id_here')")

## List analyses

In [ ]:
if trajectory_id:
    for analysis in traj.analyses:
        print(f"  {analysis.raw.get('pluginDisplayName')}: {analysis.raw.get('status')}  (id={analysis.id})")

## Analysis results as a DataFrame

`analysis.df()` returns the analysis listing rows (one row per timestep).

In [ ]:
if trajectory_id:
    analysis = traj.analyses.first()
    if analysis:
        df = analysis.df()
        print(f"Rows: {len(df)}, Columns: {list(df.columns)}")
        df.head(10)
    else:
        print('No analyses yet. Run an analysis plugin on this trajectory in Volt.')

## Per-atom data

`frame.atoms()` returns every atom of a frame as a DataFrame (id, type, x, y, z, plus any per-atom analysis columns). All pages are fetched for you.

In [ ]:
if trajectory_id and len(traj.frames):
    atoms = traj.frames.first().atoms()
    print(f"Atoms: {len(atoms)}, Columns: {list(atoms.columns)}")
    atoms.head(10)

## Plot a result

In [ ]:
import matplotlib.pyplot as plt

if trajectory_id and analysis and not df.empty:
    exclude = {'_id', 'analysisId', 'trajectoryId', 'exposureId', 'trajectoryName'}
    numeric_cols = [c for c in df.select_dtypes(include='number').columns
                    if c not in exclude and c != 'timestep']
    if 'timestep' in df.columns and numeric_cols:
        df.plot(x='timestep', y=numeric_cols[:4], subplots=True, figsize=(8, 3 * min(4, len(numeric_cols))))
        plt.tight_layout()
        plt.show()
    else:
        print(f'No numeric timeseries to plot. Columns: {list(df.columns)}')

## Download artifacts

Download the raw analysis output (data files + GLB 3D models) for offline work.

In [ ]:
# Uncomment to download the first analysis's artifacts:
# if analysis:
#     out_dir = analysis.download_artifacts(dest='./analysis_data/')
#     print(f'Saved to: {out_dir}')

## View in the Volt 3D canvas

In [ ]:
# Open the trajectory (optionally an analysis/timestep) in the Volt web viewer:
# traj.open_in_volt()
#
# Or render a local GLB you produced:
# from voltsdk import open_in_volt
# open_in_volt('model.glb')

## Quick reference

| Task | Code |
|------|------|
| Connect | `client = VoltClient.from_env()` |
| List trajectories | `client.trajectories.list()` |
| Get trajectory | `client.trajectories.get("id")` |
| List analyses | `traj.analyses.list()` |
| Analysis results | `analysis.df()` |
| Listings to CSV | `analysis.listings.to_csv("out.csv")` |
| Per-atom data | `traj.frames.first().atoms()` |
| Download artifacts | `analysis.download_artifacts(dest="./data/")` |
| Download a frame dump | `frame.download_dump()` |
| OVITO pipeline | `traj.to_ovito_pipeline()` |
| Open in Volt canvas | `traj.open_in_volt()` |